# motif-discover vs STREME: Live Benchmark

Runs both tools independently on the same ENCODE K562 ChIP-seq data and compares results.

## 1. Setup

In [ ]:
import os, subprocess

if not os.path.isdir('/content/motif-discover/.git'):
    subprocess.run('rm -rf /content/motif-discover && git clone -q https://github.com/Travis42/motif-discover.git /content/motif-discover', shell=True)

os.chdir('/content/motif-discover')
subprocess.run(['chmod', '+x', 'motif-discover', 'streme'])

n_tfs = len([f for f in os.listdir('example') if f.endswith('.fa')])
print(f'Ready: {n_tfs} TFs')
print(f'motif-discover: {os.path.getsize("motif-discover")//1024} KB')
print(f'STREME:         {os.path.getsize("streme")//1024} KB')

## 2. Run motif-discover

In [ ]:
%%time
r_ours = subprocess.run(
    ['./motif-discover', '--data', 'example/', '--ours-only', '--no-meme'],
    capture_output=True, text=True
)
print(r_ours.stdout)

## 3. Run STREME

STREME runs on the same FASTA files, same width range (6–17), same DNA alphabet.

In [ ]:
%%time
import glob, json, re

streme_results = {}
for fa_path in sorted(glob.glob('example/*_sequences.fa')):
    tf = os.path.basename(fa_path).replace('_sequences.fa', '')
    out_dir = f'/tmp/streme_{tf}'
    os.makedirs(out_dir, exist_ok=True)
    
    r = subprocess.run(
        ['./streme', '--p', fa_path, '--oc', out_dir,
         '--dna', '--minw', '6', '--maxw', '17'],
        capture_output=True, text=True, timeout=120
    )
    
    if r.returncode != 0:
        print(f'{tf}: STREME FAILED (exit {r.returncode})')
        print(r.stderr[:200])
        continue
    
    # Parse runtime from STREME output
    time_match = re.search(r'FINALTIME:\s*([\d.]+)', r.stderr)
    streme_time = float(time_match.group(1)) if time_match else 0
    
    streme_results[tf] = {'out_dir': out_dir, 'time': streme_time}
    print(f'{tf}: {streme_time:.1f}s')

print(f'\nSTREME completed: {len(streme_results)}/{n_tfs} TFs')

## 4. Score and Compare

Both tools' PWMs are scored by our benchmark binary against the same negatives (dinucleotide-shuffled sequences), producing AUROC for each.

In [ ]:
# Run our binary in full comparison mode (it runs motif-discover AND 
# scores STREME's output from the /tmp/streme_TF/ directories)
r_both = subprocess.run(
    ['./motif-discover', '--data', 'example/', '--no-meme',
     '--streme-bin', './streme'],
    capture_output=True, text=True
)

import pandas as pd

rows = []
for line in r_both.stdout.strip().split('\n'):
    parts = line.split('\t')
    if len(parts) >= 5 and parts[0] != 'TF' and parts[4] in ('ours', 'streme'):
        rows.append({
            'TF': parts[0],
            'Width': int(parts[1]),
            'AUROC': float(parts[2]),
            'Time_s': float(parts[3]),
            'Tool': 'motif-discover' if parts[4] == 'ours' else 'STREME',
        })

df = pd.DataFrame(rows)
print(f'Total rows parsed: {len(df)}')
if len(df) == 0:
    print('No data rows found. Stderr:')
    print(r_both.stderr[:500])
else:
    print(df.to_string(index=False))

## 5. Visualize

In [ ]:
if len(df) > 0:
    ours = df[df['Tool'] == 'motif-discover'].set_index('TF')
    streme = df[df['Tool'] == 'STREME'].set_index('TF')
    common = sorted(ours.index.intersection(streme.index))

    if len(common) > 0:
        import matplotlib.pyplot as plt
        import numpy as np

        MD_COLOR = '#2166AC'
        ST_COLOR = '#D6604D'

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        # Panel A: Per-TF AUROC
        ax = axes[0]
        x = np.arange(len(common))
        w = 0.35
        ax.barh(x - w/2, ours.loc[common, 'AUROC'], w, color=MD_COLOR, alpha=0.8, label='motif-discover')
        ax.barh(x + w/2, streme.loc[common, 'AUROC'], w, color=ST_COLOR, alpha=0.8, label='STREME')
        ax.set_yticks(x)
        ax.set_yticklabels(common, fontsize=9)
        ax.set_xlabel('AUROC')
        ax.set_title('Per-TF AUROC')
        ax.axvline(0.5, color='gray', linestyle='--', alpha=0.3)
        ax.legend()

        # Panel B: Scatter
        ax = axes[1]
        o = ours.loc[common, 'AUROC'].values
        s = streme.loc[common, 'AUROC'].values
        colors = np.where(o > s, MD_COLOR, ST_COLOR)
        ax.scatter(s, o, alpha=0.7, s=60, c=colors)
        ax.plot([0.3, 1.0], [0.3, 1.0], 'k--', alpha=0.3)
        ax.set_xlabel('STREME AUROC')
        ax.set_ylabel('motif-discover AUROC')
        ax.set_title(f'Per-TF (n={len(common)})')
        for tf in common:
            d = ours.loc[tf, 'AUROC'] - streme.loc[tf, 'AUROC']
            if abs(d) > 0.1:
                ax.annotate(tf, (streme.loc[tf, 'AUROC'], ours.loc[tf, 'AUROC']), fontsize=8)

        # Panel C: Speed
        ax = axes[2]
        md_t = ours.loc[common, 'Time_s'].mean()
        st_t = streme.loc[common, 'Time_s'].mean()
        bars = ax.bar(['motif-discover', 'STREME'], [md_t, st_t],
                      color=[MD_COLOR, ST_COLOR], edgecolor='black', width=0.5)
        for bar, t in zip(bars, [md_t, st_t]):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                    f'{t:.2f}s', ha='center', fontsize=12, fontweight='bold')
        ax.set_ylabel('Time per TF (seconds)')
        ax.set_title('Speed')

        plt.tight_layout()
        plt.show()

        print(f'\nmotif-discover: AUROC={ours.loc[common, "AUROC"].mean():.4f} ({md_t:.2f}s/TF)')
        print(f'STREME:         AUROC={streme.loc[common, "AUROC"].mean():.4f} ({st_t:.2f}s/TF)')
        print(f'Speedup: {st_t/md_t:.1f}x')
    else:
        print('No common TFs between tools')
else:
    print('No results to visualize')

---

Live benchmark on this machine. Full 132-TF results and paper: [github.com/Travis42/motif-discover](https://github.com/Travis42/motif-discover)